# 36 · Pause a graph and resume with human input

A graph can emit RequestInput without calling a model. The application correlates the human response with the interrupt's function-call ID and resumes the same invocation.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Zero model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk import Workflow, Event
from google.adk.events import RequestInput
from google.adk.apps import App, ResumabilityConfig

def request_delivery_slot():
    yield RequestInput(message="Choose morning or evening delivery for O1001.",
                       payload={"order_id": "O1001", "allowed_slots": ["morning", "evening"]})

def confirm_slot(node_input: dict):
    slot = node_input.get("slot")
    if slot not in {"morning", "evening"}:
        raise ValueError("Choose morning or evening")
    return Event(message=f"Demo delivery preference recorded: {slot}. No carrier was contacted.")

workflow = Workflow(name="delivery_preference", edges=[("START", request_delivery_slot, confirm_slot)])
app = App(name="delivery_app", root_agent=workflow, resumability_config=ResumabilityConfig(is_resumable=True))
lab = await make_lab(app=app)
events = await ask(lab, "Choose a delivery slot.")
calls = [call for event in events for call in event.get_function_calls()]
assert calls
interrupt = calls[-1]
invocation_id = events[-1].invocation_id
print("Waiting for human input:", interrupt.args)


In [ ]:
# This value represents a selection supplied by the human UI.
HUMAN_CHOICE = "evening"
events = await ask(lab, invocation_id=invocation_id,
    parts=[types.Part(function_response=types.FunctionResponse(
        id=interrupt.id, name=interrupt.name, response={"slot": HUMAN_CHOICE}))])
assert "evening" in final_text(events)


## Try it

Try an invalid slot and handle validation before submitting the UI response. Persist sessions in SQLite to retain the pause across kernel restarts.


## Reference

[Official ADK documentation](https://adk.dev/graphs/human-input/). Shared configuration: `config.json`. No environment activation or login cells are needed.
